# 🧠 El perceptrón: en Python y en Keras

Un **perceptrón** es la red neuronal más sencilla que existe: **una sola neurona**.

En este cuaderno:
1. Haremos una neurona **a mano**, en Python normal.
2. Haremos que **aprenda sola** sus pesos y su sesgo.
3. Haremos lo mismo con **Keras**, en pocas líneas.

📄 Antes conviene leer el PDF *¿Cómo funciona una red neuronal?*

---
# PARTE 1 · El perceptrón en Python (sin librerías)

## 1.1 La neurona de la playa
Es la misma neurona del PDF. Tiene 3 entradas (1 = sí, 0 = no):
- ¿Hace sol?
- ¿Tengo tiempo?
- ¿Está lejos?

Una neurona hace 3 cosas:
1. Multiplica cada entrada por su **peso**
2. Lo suma todo y le suma el **sesgo**
3. Si el total es **mayor que 0** → dice **1 (SÍ)**. Si no → dice **0 (NO)**

In [ ]:
def neurona(entradas, pesos, sesgo):
    total = 0
    for entrada, peso in zip(entradas, pesos):
        total = total + entrada * peso       # 1. multiplicar y sumar
    total = total + sesgo                    # 2. sumar el sesgo

    if total > 0:                            # 3. decidir
        return 1
    else:
        return 0

Ponemos los pesos y el sesgo del PDF:

In [ ]:
pesos = [0.6, 0.4, -0.5]    # sol, tiempo, lejos
sesgo = -0.5

Probamos con distintos días:

In [ ]:
print("Sol, tiempo y cerca  →", neurona([1, 1, 0], pesos, sesgo))
print("Sol, tiempo y lejos  →", neurona([1, 1, 1], pesos, sesgo))
print("Sin sol, tiempo, cerca →", neurona([0, 1, 0], pesos, sesgo))
print("Sol, sin tiempo, cerca →", neurona([1, 0, 0], pesos, sesgo))

🧪 **Prueba:** cambia el sesgo a `0.5`. ¿Ahora la neurona dice SÍ más veces o menos? ¿Por qué?

---
## 1.2 El perceptrón aprende solo
Ahora **no le damos** los pesos ni el sesgo. Empiezan en **0** y la neurona tiene que **descubrirlos** con ejemplos.

**Problema:** voy a la playa **solo si** hace sol **y** tengo tiempo.

| ¿Sol? | ¿Tiempo? | ¿Voy? |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

In [ ]:
ejemplos   = [[0, 0], [0, 1], [1, 0], [1, 1]]
respuestas = [0, 0, 0, 1]

### ¿Cómo aprende?
Para cada ejemplo:
1. La neurona **predice**.
2. Calculamos el **error** = respuesta correcta − predicción.
3. Si se ha equivocado, **corregimos** un poquito los pesos y el sesgo:

```
peso  = peso  + velocidad × error × entrada
sesgo = sesgo + velocidad × error
```

💡 Si el error es 0 (ha acertado), no cambia nada.

In [ ]:
pesos = [0.0, 0.0]      # empieza sin saber nada
sesgo = 0.0
velocidad = 0.1         # tamaño de cada corrección

for epoca in range(10):
    fallos = 0
    for entradas, correcta in zip(ejemplos, respuestas):
        prediccion = neurona(entradas, pesos, sesgo)     # 1. predecir
        error = correcta - prediccion                    # 2. calcular el error

        for i in range(len(pesos)):                      # 3. corregir
            pesos[i] = pesos[i] + velocidad * error * entradas[i]
        sesgo = sesgo + velocidad * error

        if error != 0:
            fallos = fallos + 1

    print(f"Época {epoca + 1}:  fallos = {fallos}   pesos = {[round(p, 2) for p in pesos]}   sesgo = {round(sesgo, 2)}")

👀 Al principio falla. En pocas épocas los fallos llegan a **0**: ¡ha aprendido!

Comprobamos con los 4 casos:

In [ ]:
for entradas, correcta in zip(ejemplos, respuestas):
    print(entradas, "→ la neurona dice", neurona(entradas, pesos, sesgo), "  (correcto:", correcta, ")")

✅ La neurona ha encontrado **ella sola** unos pesos y un sesgo que funcionan. Fíjate en que el sesgo es **negativo**: la neurona es "difícil de convencer" y solo dice SÍ cuando las dos entradas son 1.

---
# PARTE 2 · El mismo perceptrón con Keras

Keras hace todo lo anterior por nosotros: crear la neurona, predecir, calcular el error y corregir.

In [ ]:
import numpy as np
from tensorflow import keras

Los mismos ejemplos, pero en formato `numpy`:

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 0, 0, 1], dtype=float)

## 2.1 Crear la neurona
- `keras.Input(shape=(2,))` → 2 entradas (sol y tiempo)
- `Dense(1, activation="sigmoid")` → **1 neurona**. La `sigmoid` da un número entre 0 y 1 (una probabilidad).

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(2,)),
    keras.layers.Dense(1, activation="sigmoid")
])
modelo.compile(loss="binary_crossentropy",
               optimizer=keras.optimizers.Adam(0.1),
               metrics=["accuracy"])

## 2.2 Entrenar

In [ ]:
modelo.fit(X, y, epochs=200, verbose=0)
print("Entrenada ✅")

## 2.3 Probar
La red da una **probabilidad**. Si es mayor que 0.5 → SÍ.

In [ ]:
probabilidades = modelo.predict(X, verbose=0)

for entradas, p in zip(X, probabilidades):
    respuesta = 1 if p[0] > 0.5 else 0
    print(entradas, f"→ {p[0]:.2f}  → dice {respuesta}")

## 2.4 Los pesos y el sesgo que ha encontrado Keras

In [ ]:
pesos_keras, sesgo_keras = modelo.get_weights()
print("Pesos:", pesos_keras.flatten())
print("Sesgo:", sesgo_keras)

👀 Igual que en Python: dos pesos **positivos** y un sesgo **negativo**.

## 📝 Comparación
| | Python a mano | Keras |
|---|---|---|
| Crear la neurona | función `neurona()` | `Dense(1)` |
| Activación | escalón (`if total > 0`) | `sigmoid` |
| Aprender | bucle que corrige los pesos | `modelo.fit()` |
| Líneas de código | ~20 | ~8 |

Keras hace lo mismo, pero **más rápido y con menos código**. Y con Keras podemos poner **miles de neuronas** sin escribir más bucles.

---
# ✏️ EJERCICIO
Ahora voy a la playa si hace sol **o** tengo tiempo (basta con una de las dos).

| ¿Sol? | ¿Tiempo? | ¿Voy? |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

1. Entrena el perceptrón **en Python** con estos datos.
2. Entrena el perceptrón **en Keras** con estos datos.
3. Compara los sesgos: ¿es más fácil o más difícil de convencer que antes?

In [ ]:
# ✏️ 1. Perceptrón en Python
respuestas_o = [0, 1, 1, 1]

In [ ]:
# ✏️ 2. Perceptrón en Keras

✏️ **3. Respuesta:** ...

---
<details>
<summary>👀 <b>Ver la solución</b> (haz clic)</summary>

**1. Python**
```python
respuestas_o = [0, 1, 1, 1]
pesos = [0.0, 0.0]
sesgo = 0.0
for epoca in range(10):
    for entradas, correcta in zip(ejemplos, respuestas_o):
        error = correcta - neurona(entradas, pesos, sesgo)
        for i in range(len(pesos)):
            pesos[i] = pesos[i] + 0.1 * error * entradas[i]
        sesgo = sesgo + 0.1 * error
print(pesos, sesgo)
for entradas in ejemplos:
    print(entradas, "→", neurona(entradas, pesos, sesgo))
```

**2. Keras**
```python
y_o = np.array([0, 1, 1, 1], dtype=float)
modelo_o = keras.Sequential([keras.Input(shape=(2,)), keras.layers.Dense(1, activation="sigmoid")])
modelo_o.compile(loss="binary_crossentropy", optimizer=keras.optimizers.Adam(0.1), metrics=["accuracy"])
modelo_o.fit(X, y_o, epochs=200, verbose=0)
print(modelo_o.predict(X, verbose=0))
print(modelo_o.get_weights())
```

**3.** El sesgo ahora es **casi 0 o menos negativo** que antes: la neurona es **más fácil de convencer**, porque con una sola entrada en 1 ya dice SÍ.
</details>